# Customer Segmentation using RFM Analysis & K-Means Clustering

**Goal:** Segment e-commerce customers into meaningful groups based on their purchasing behaviour (Recency, Frequency, Monetary value), so marketing can target each group differently.

**Dataset:** `online_retail_sample.csv` — a transaction-level e-commerce dataset in the same shape as the UCI/Kaggle *Online Retail* dataset (InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, CustomerID, Country).

> If you want to use the **real** dataset instead of the sample included here: search *"Online Retail Dataset"* on Kaggle (originally from the UCI Machine Learning Repository), download `Online Retail.xlsx` / `.csv`, and point `DATA_PATH` below at it. The rest of the notebook will run unchanged since the column names match.

**Tech stack:** Python, pandas, scikit-learn (KMeans), matplotlib, seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

DATA_PATH = 'online_retail_sample.csv'  # swap for the real Kaggle/UCI file if you have it

## 1. Load Dataset & Inspect Structure

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['InvoiceDate'])
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

### Handle missing values & inconsistent data
Common issues in this kind of transactional data:
- **Missing `CustomerID`** → these are guest/unidentified transactions and can't be attributed to a customer, so we drop them.
- **Missing `Description`** → doesn't affect customer-level RFM, but we clean it up anyway.
- **Cancelled orders** → InvoiceNo starting with `C` and/or negative `Quantity`. These aren't genuine purchases, so we remove them for the segmentation (a returns-analysis project might keep them separately).
- **Zero/negative `UnitPrice`** → data entry errors, not real revenue, so we drop those rows.
- **Duplicate rows** → exact duplicate transactions get removed.

In [ ]:
print("Missing values per column:\n", df.isna().sum())
print("\nCancelled invoices (start with 'C'):", df['InvoiceNo'].astype(str).str.startswith('C').sum())
print("Negative/zero quantity rows:", (df['Quantity'] <= 0).sum())
print("Zero/negative unit price rows:", (df['UnitPrice'] <= 0).sum())
print("Exact duplicate rows:", df.duplicated().sum())

In [ ]:
df_clean = df.copy()

# Drop rows with no CustomerID -- can't build a customer profile without one
df_clean = df_clean.dropna(subset=['CustomerID'])

# Remove cancellations (InvoiceNo starting with 'C') and non-positive quantities
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
df_clean = df_clean[df_clean['Quantity'] > 0]

# Remove rows with non-positive unit price (data entry errors / free items)
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Drop exact duplicate rows
df_clean = df_clean.drop_duplicates()

# Fill any remaining missing descriptions with a placeholder (doesn't affect RFM)
df_clean['Description'] = df_clean['Description'].fillna('UNKNOWN ITEM')

df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

print("Rows before cleaning:", len(df))
print("Rows after cleaning: ", len(df_clean))
print("Unique customers:     ", df_clean['CustomerID'].nunique())

In [ ]:
# TotalPrice per line item = Quantity * UnitPrice
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean.head()

## 2. Descriptive Statistics

In [ ]:
order_value = df_clean.groupby('InvoiceNo')['TotalPrice'].sum()
avg_purchase_value = order_value.mean()

purchase_freq = df_clean.groupby('CustomerID')['InvoiceNo'].nunique()
avg_purchase_frequency = purchase_freq.mean()

clv_per_customer = df_clean.groupby('CustomerID')['TotalPrice'].sum()
avg_clv = clv_per_customer.mean()

print(f"Average purchase (order) value:      £{avg_purchase_value:,.2f}")
print(f"Average purchase frequency/customer: {avg_purchase_frequency:.2f} orders")
print(f"Average customer lifetime value:     £{avg_clv:,.2f}")
print(f"Median customer lifetime value:      £{clv_per_customer.median():,.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(order_value, bins=40, ax=axes[0], color='#4C72B0')
axes[0].set_title('Distribution of Order Value')
axes[0].set_xlabel('Order Value (£)')

sns.histplot(clv_per_customer, bins=40, ax=axes[1], color='#55A868')
axes[1].set_title('Distribution of Customer Lifetime Value')
axes[1].set_xlabel('Total Spend per Customer (£)')
plt.tight_layout()
plt.show()

## 3. Feature Selection — RFM Analysis
We build the three classic behavioural features for customer segmentation:

- **Recency (R)** — days since the customer's last purchase (lower = more recently active)
- **Frequency (F)** — number of distinct orders placed
- **Monetary (M)** — total amount spent

In [ ]:
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print("Snapshot (reference) date:", snapshot_date.date())

rfm = df_clean.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

rfm.head()

In [ ]:
rfm[['Recency', 'Frequency', 'Monetary']].describe()

## 4. Data Normalisation / Standardisation
K-Means uses distance, so features on very different scales (days vs. order count vs. currency) must be standardised first.

In [ ]:
features = ['Recency', 'Frequency', 'Monetary']
X = rfm[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=features)
X_scaled_df.describe().round(2)

## 5. K-Means Clustering — Elbow Method

In [ ]:
inertia = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(7, 4.5))
plt.plot(list(k_range), inertia, marker='o')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Inertia (within-cluster sum of squares)')
plt.title('Elbow Method for Optimal K')
plt.xticks(list(k_range))
plt.show()

Looking at the plot, inertia drops sharply up to **K = 4**, after which the improvement flattens out — that's our elbow, so we'll go with **K = 4** clusters.

In [ ]:
OPTIMAL_K = 4

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(X_scaled)

rfm['Cluster'].value_counts().sort_index()

## 6. Visualise Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster',
                 palette='Set2', s=60, alpha=0.8, ax=axes[0])
axes[0].set_title('Recency vs Monetary')

sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='Cluster',
                 palette='Set2', s=60, alpha=0.8, ax=axes[1])
axes[1].set_title('Frequency vs Monetary')

plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(data=rfm, x='Recency', y='Frequency', hue='Cluster',
                 palette='Set2', s=60, alpha=0.8)
plt.title('Recency vs Frequency')
plt.show()

## 7. Cluster Profiling

In [ ]:
cluster_profile = rfm.groupby('Cluster')[features].mean().round(1)
cluster_profile['CustomerCount'] = rfm['Cluster'].value_counts().sort_index()
cluster_profile

In [ ]:
# Rank clusters to make labeling systematic rather than eyeballed
profile = cluster_profile.copy()
profile['Recency_rank'] = profile['Recency'].rank()          # low recency (recent) = rank 1 = good
profile['Frequency_rank'] = profile['Frequency'].rank(ascending=False)  # high frequency = rank 1 = good
profile['Monetary_rank'] = profile['Monetary'].rank(ascending=False)    # high monetary = rank 1 = good
profile['Score'] = profile[['Recency_rank', 'Frequency_rank', 'Monetary_rank']].mean(axis=1)
profile.sort_values('Score')

### Naming the segments
Based on the mean R/F/M values per cluster:

| Cluster pattern | Label | Description |
|---|---|---|
| Low Recency, High Frequency, High Monetary | **Champions** | Recent, frequent, big spenders — the most valuable customers |
| Moderate Recency, Moderate Frequency/Monetary | **Loyal / Regular Customers** | Steady, dependable repeat buyers |
| High Recency, Low Frequency, Low-Moderate Monetary | **At Risk / Lapsing** | Used to buy, haven't been back in a while |
| High Recency, Very Low Frequency, Low Monetary | **New / One-time / Lost** | Bought once or barely engaged, mostly inactive |

(Exact mapping of cluster numbers → labels below is generated dynamically from the ranked data so it stays correct even if you rerun with different data.)

In [ ]:
def label_cluster(row):
    if row['Score'] <= profile['Score'].quantile(0.25):
        return 'Champions'
    elif row['Score'] <= profile['Score'].median():
        return 'Loyal Customers'
    elif row['Score'] <= profile['Score'].quantile(0.75):
        return 'At Risk'
    else:
        return 'New / Lost'

profile['Label'] = profile.apply(label_cluster, axis=1)
cluster_labels = profile['Label'].to_dict()
rfm['Segment'] = rfm['Cluster'].map(cluster_labels)

print(cluster_labels)
rfm.groupby('Segment')[features].mean().round(1)

## 8. Customers per Cluster

In [ ]:
order = rfm['Segment'].value_counts().index
plt.figure(figsize=(7.5, 4.5))
sns.countplot(data=rfm, x='Segment', hue='Segment', order=order, palette='Set2', legend=False)
plt.title('Number of Customers per Segment')
plt.xlabel('')
plt.ylabel('Customer Count')
plt.xticks(rotation=15)
plt.show()

## 9. Insights & Recommended Marketing Actions

**Champions** (recent, frequent, high spend)
- Reward with a VIP/loyalty tier, early access to new products, and exclusive discounts.
- Ask for reviews/referrals — they're your best advocates.
- Avoid generic discount blasts; they'll buy anyway, so don't erode margin unnecessarily.

**Loyal Customers** (steady repeat buyers, moderate spend)
- Upsell/cross-sell complementary products based on past purchase history.
- Bundle offers and "customers like you also bought" recommendations to lift order value.
- Enroll in a points/loyalty programme to nudge them toward Champion status.

**At Risk** (haven't purchased recently, used to buy more)
- Win-back email campaigns with a time-limited incentive (e.g. "we miss you" 15% off).
- Personalised recommendations based on their historical favourite categories.
- Survey to find out why engagement dropped (price, product fit, service issue).

**New / Lost** (one-off or long-inactive, low spend)
- For genuinely *new* customers: onboarding email series, first-repeat-purchase incentive.
- For *lost* customers: low-cost reactivation campaign; if no response after 1–2 attempts, deprioritize spend on this group to protect marketing ROI.
- A/B test subject lines and offer types to see what (if anything) re-engages them cheaply.

### Summary table

In [ ]:
summary = rfm.groupby('Segment').agg(
    Customers=('CustomerID', 'count'),
    Avg_Recency_days=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).round(1).sort_values('Avg_Monetary', ascending=False)

summary

In [ ]:
# Save final labeled customer table for use in downstream marketing tools / dashboards
rfm.to_csv('customer_segments_output.csv', index=False)
print("Saved: customer_segments_output.csv")
rfm.head()